# HW3 — Two retrievers and a model: what RAG actually sends

**Name:**
**Student ID:**
**Group:**
**Repository:**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl = 8.33**

**Query terms: How, does, RAG, retrieve, information**

**TF, DF and IDF of each query term:**

| Query term  | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|-------------|---------:|---------:|---------:|----------:|-------:|
| how         |        0 |        0 |        0 |         0 | 1.9459 |
| does        |        0 |        0 |        0 |         0 | 1.9459 |
| rag         |        0 |        0 |        0 |         0 | 1.9459 |
| retrieve    |        0 |        0 |        0 |         0 | 1.9459 |
| information |        0 |        0 |        0 |         0 | 1.9459 |


**BM25 of each document** (write out the sum for every document, one line per term that contributes):

**IDF calculation:**

All query terms have document frequency n(t) = 0.

$$
idf(t)=\ln\left(\frac{N-n(t)+0.5}{n(t)+0.5}\right)
$$

$$
=\ln\left(\frac{3-0+0.5}{0+0.5}\right)
$$

$$
=\ln\left(\frac{3.5}{0.5}\right)
=\ln(7)=1.9459
$$

**BM25 of each document:**

The BM25 formula is:

$$
BM25(q,d)=\sum_{t\in q}idf(t)\cdot
\frac{f(t,d)(k1+1)}
{f(t,d)+k1(1-b+b|d|/avgdl)}
$$

No query term appears exactly in any document.

In particular, "retrieve" does not exactly match "retrieval" or "retrieves".

Therefore, every term frequency is zero, and every term contributes zero to the BM25 score.

**D1:**

$$
BM25(D1)=0+0+0+0+0=0
$$

**D2:**

$$
BM25(D2)=0+0+0+0+0=0
$$

**D3:**

$$
BM25(D3)=0+0+0+0+0=0
$$
>

| Document | BM25 Score | Rank |
|---|-----------:|-----:|
| D1 |          0 |    1 |
| D2 |          0 |    1 |
| D3 |          0 |    1 |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, model, can, gener, text, and, answer, question | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, document, before, gener, an, answer | 10 |
| D3 | embedding, represent, text, as, numerical, vector | 6 |

**avgdl = 8.33**

**Stemmed query terms: how, doe, rag, retriev, inform**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|------------|---------:|---------:|---------:|----------:|-------:|
| how        |        0 |        0 |        0 |         0 | 1.9459 |
| doe        |        0 |        0 |        0 |         0 | 1.9459 |
| rag        |        0 |        0 |        0 |         0 | 1.9459 |
| retriev    |        0 |        2 |        0 |         1 | 0.5108 |
| inform     |        0 |        0 |        0 |         0 | 1.9459 |

**BM25 of each document**, worked:

**IDF calculations:**

For "how", "doe", "rag", and "inform", document frequency is 0:

$$
idf(t)=\ln\left(\frac{3-0+0.5}{0+0.5}\right)
$$

$$
=\ln(7)=1.9459
$$

For "retriev", document frequency is 1 because it occurs only in D2:

$$
idf(retriev)=\ln\left(\frac{3-1+0.5}{1+0.5}\right)
$$

$$
=\ln\left(\frac{2.5}{1.5}\right)
$$

$$
=\ln(5/3)=0.5108256
$$

The term "retriev" occurs twice in D2:

- retrieval -> retriev
- retrieves -> retriev

Therefore:

$$
f(retriev,D2)=2
$$

**BM25 of each document, worked:**

BM25 formula:

$$
BM25(q,d)=\sum_{t\in q}idf(t)\cdot
\frac{f(t,d)(k1+1)}
{f(t,d)+k1(1-b+b|d|/avgdl)}
$$

**D1:**

No stemmed query terms occur in D1.

$$
BM25(D1)=0+0+0+0+0=0
$$

**D2:**

Only "retriev" contributes to the score.

Known values:

- f(retriev,D2) = 2
- idf(retriev) = 0.5108256
- k1 = 1.5
- b = 0.75
- |D2| = 10
- avgdl = 25/3

First, calculate document length normalization:

$$
1-b+b\frac{|D2|}{avgdl}
$$

$$
=1-0.75+0.75\cdot\frac{10}{25/3}
$$

$$
=0.25+0.75\cdot1.2
$$

$$
=0.25+0.9=1.15
$$

Calculate the denominator:

$$
f+k1(1-b+b|D2|/avgdl)
$$

$$
=2+1.5(1.15)
$$

$$
=2+1.725=3.725
$$

Calculate the numerator without IDF:

$$
f(k1+1)=2(1.5+1)
$$

$$
=2(2.5)=5
$$

Substitute into BM25:

$$
BM25(D2)=0.5108256\cdot\frac{5}{3.725}
$$

$$
=0.5108256\cdot1.3422819
$$

$$
\boxed{BM25(D2)\approx0.685672}
$$

All other query terms contribute 0.

**D3:**

No stemmed query terms occur in D3.

$$
BM25(D3)=0+0+0+0+0=0
$$
>

| Document | BM25 Score | Rank |
|---|-----------:|-----:|
| D1 |          0 |    2 |
| D2 |   0.685672 |    1 |
| D3 |          0 |    2 |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

>In the first case, there were no exact matches between the query and any of the documents, so BM25 gave all three documents a score of 0 and could not rank them meaningfully.
>
>In the second case, after applying stemming, a match was found in D2. As a result, BM25 ranked D2 first, while D1 and D3 still had scores of 0 and remained tied.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```
**Cosine similarity formula:**

$$
cos(q,d)=\frac{q\cdot d}{||q||\cdot||d||}
$$

**Step 1 — Query magnitude**

$$
||q||=\sqrt{0.8^2+0.2^2+0.5^2}
$$

$$
=\sqrt{0.64+0.04+0.25}
$$

$$
=\sqrt{0.93}\approx0.964365
$$

**Step 2 — Document magnitudes**

D1:

$$
||D1||=\sqrt{0.7^2+0.1^2+0.4^2}
$$

$$
=\sqrt{0.49+0.01+0.16}
$$

$$
=\sqrt{0.66}\approx0.812404
$$

D2:

$$
||D2||=\sqrt{0.9^2+0.3^2+0.6^2}
$$

$$
=\sqrt{0.81+0.09+0.36}
$$

$$
=\sqrt{1.26}\approx1.122497
$$

D3:

$$
||D3||=\sqrt{0.1^2+0.8^2+0.2^2}
$$

$$
=\sqrt{0.01+0.64+0.04}
$$

$$
=\sqrt{0.69}\approx0.830662
$$

**Step 3 — Dot products**

Query and D1:

$$
q\cdot D1=0.8(0.7)+0.2(0.1)+0.5(0.4)
$$

$$
=0.56+0.02+0.20=0.78
$$

Query and D2:

$$
q\cdot D2=0.8(0.9)+0.2(0.3)+0.5(0.6)
$$

$$
=0.72+0.06+0.30=1.08
$$

Query and D3:

$$
q\cdot D3=0.8(0.1)+0.2(0.8)+0.5(0.2)
$$

$$
=0.08+0.16+0.10=0.34
$$

**Step 4 — Cosine similarity**

Query and D1:

$$
cos(q,D1)=\frac{0.78}{\sqrt{0.93}\sqrt{0.66}}
$$

$$
=\frac{0.78}{\sqrt{0.6138}}
\approx0.995591
$$

Query and D2:

$$
cos(q,D2)=\frac{1.08}{\sqrt{0.93}\sqrt{1.26}}
$$

$$
=\frac{1.08}{\sqrt{1.1718}}
\approx0.997693
$$

Query and D3:

$$
cos(q,D3)=\frac{0.34}{\sqrt{0.93}\sqrt{0.69}}
$$

$$
=\frac{0.34}{\sqrt{0.6417}}
\approx0.424437
$$

| Pair       | Dot product (worked)      |  ‖Query‖ |      ‖D‖ |   Cosine |
|------------|---------------------------|---------:|---------:|---------:|
| Query · D1 | 0.56 + 0.02 + 0.20 = 0.78 | 0.964365 | 0.812404 | 0.995591 |
| Query · D2 | 0.72 + 0.06 + 0.30 = 1.08 | 0.964365 | 1.122497 | 0.997693 |
| Query · D3 | 0.08 + 0.16 + 0.10 = 0.34 | 0.964365 | 0.830662 | 0.424437 |

| Document | Cosine Similarity | Rank |
|----------|------------------:|-----:|
| D1       |          0.995591 |    2 |
| D2       |          0.997693 |    1 |
| D3       |          0.424437 |    3 |

**Which document is most similar to the query?**

>D2 is the most similar document, with a cosine similarity of approximately 0.997693.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

>The ranking is different from BM25 without stemming, where all three documents received a score of 0 and were tied.
>
>With stemming, BM25 also ranks D2 first, but D1 and D3 remain tied. Cosine similarity produces a complete ranking: D2 > D1 > D3.


**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

>BM25 depends on matching words. In the query "How does RAG retrieve information?", the word "retrieve" does not exactly match "retrieval" or "retrieves" in D2 without stemming.
>
>Embedding retrieval uses numerical representations and can capture semantic relationships even when words are different. In the supplied vectors, D2 is closest to the query, and D1 is also very close despite having no exact query terms. This explains why the two approaches can produce different rankings.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [38]:
import json
from pathlib import Path

data_dir = Path("data")
corpus_path = data_dir / "corpus.jsonl"
queries_path = data_dir / "queries.json"

with open(corpus_path, "r", encoding="utf-8") as f:
    documents = [json.loads(line) for line in f if line.strip()]

with open(queries_path, "r", encoding="utf-8") as f:
    queries_data = json.load(f)

queries = (
    queries_data["queries"]
    if isinstance(queries_data, dict)
    else queries_data
)

print(f"Loaded documents: {len(documents)}")
print(f"Loaded queries: {len(queries)}")

assert len(documents) == 74
assert len(queries) == 8

print("\nFirst document:")
print(json.dumps(documents[0], indent=2, ensure_ascii=False))

print("\nFirst query:")
print(json.dumps(queries[0], indent=2, ensure_ascii=False))


Loaded documents: 74
Loaded queries: 8

First document:
{
  "id": "KZ-01",
  "source": "wikipedia",
  "title": "Kazakhstan (part 1)",
  "lang": "en",
  "url": "https://en.wikipedia.org/wiki/Kazakhstan",
  "text": "Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situated primarily in Central Asia, with a portion of its territory extending into Eastern Europe. It borders Russia to the north and west, China to the east, Kyrgyzstan to the southeast, Uzbekistan to the south, and Turkmenistan to the southwest, and it has a coastline along the Caspian Sea. The capital is Astana, and the country's largest city and principal cultural and economic center is Almaty, which served as the capital until 1997."
}

First query:
{
  "id": "Q-01",
  "text": "What is form GR-305 used for?",
  "relevant": [
    "GO-08",
    "GO-12",
    "GO-23"
  ]
}


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>I convert all text to lowercase and use the regular expression `\b\w+\b` for tokenization. I apply Snowball stemming and remove common English and Russian stop words to reduce unnecessary tokens. However, I keep negations and important conditional words such as "not", "no", "only", "before", and "after", because they can change the meaning of the text. The same preprocessing is applied to both documents and queries. Other tokens are kept unchanged.


In [39]:
import os
import re
import nltk
from pathlib import Path
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

nltk_dir = Path(r"E:\hm3\nltk_data")
nltk_dir.mkdir(parents=True, exist_ok=True)

os.environ["NLTK_DATA"] = str(nltk_dir)
nltk.data.path[:] = [str(nltk_dir)]

nltk.download(
    "stopwords",
    download_dir=str(nltk_dir),
    quiet=True,
    raise_on_error=True
)

stem_en = SnowballStemmer("english")
stem_ru = SnowballStemmer("russian")

stop_en = set(stopwords.words("english"))
stop_ru = set(stopwords.words("russian"))

stop_en -= {
    "not", "no", "nor", "only", "before",
    "after", "without", "except", "until"
}

stop_ru -= {
    "не", "нет", "ни", "без", "только",
    "до", "после", "кроме"
}


def tokenize(text):
    words = re.findall(r"\b\w+\b", text.lower())
    tokens = []

    for word in words:
        if re.fullmatch(r"[a-z]+", word):
            if word not in stop_en:
                tokens.append(stem_en.stem(word))

        elif re.fullmatch(r"[а-яё]+", word):
            if word not in stop_ru:
                tokens.append(stem_ru.stem(word))

        else:
            tokens.append(word)

    return tokens


text = "Retrieval-Augmented Generation retrieves relevant documents before generating an answer."

print("Tokens:", tokenize(text))


Tokens: ['retriev', 'augment', 'generat', 'retriev', 'relev', 'document', 'befor', 'generat', 'answer']


In [40]:

from rank_bm25 import BM25Okapi

tokenized_corpus = [
    tokenize(doc["text"]) for doc in documents
]

bm25 = BM25Okapi(
    tokenized_corpus,
    k1=1.5,
    b=0.75
)

print(f"Indexed documents: {len(tokenized_corpus)}")

print("\nOriginal text:")
print(documents[0]["text"][:300])

print("\nTokenized text:")
print(tokenized_corpus[0][:40])


Indexed documents: 74

Original text:
Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situated primarily in Central Asia, with a portion of its territory extending into Eastern Europe. It borders Russia to the north and west, China to the east, Kyrgyzstan to the southeast, Uzbekistan to the south, and Turkmeni

Tokenized text:
['kazakhstan', 'offici', 'republ', 'kazakhstan', 'landlock', 'countri', 'situat', 'primarili', 'central', 'asia', 'portion', 'territori', 'extend', 'eastern', 'europ', 'border', 'russia', 'north', 'west', 'china', 'east', 'kyrgyzstan', 'southeast', 'uzbekistan', 'south', 'turkmenistan', 'southwest', 'coastlin', 'along', 'caspian', 'sea', 'capit', 'astana', 'countri', 'largest', 'citi', 'princip', 'cultur', 'econom', 'center']


In [41]:
def search_bm25(query, top_k=3):
    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: (-scores[i], i)
    )[:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        doc = documents[idx]

        first_line = next(
            (line.strip() for line in doc["text"].splitlines()
             if line.strip()),
            ""
        )

        results.append({
            "rank": rank,
            "id": doc["id"],
            "first_line": first_line,
            "score": float(scores[idx])
        })

    return results

In [42]:
bm25_results = {}

for q in queries:
    query_id = q["id"]
    query_text = q["text"]
    relevant_ids = set(q["relevant"])

    results = search_bm25(query_text, top_k=3)
    bm25_results[query_id] = results

    print(f"\n{'=' * 65}")
    print(f"{query_id}: {query_text}")
    print(f"{'=' * 65}")

    for result in results:
        print(f"Rank: {result['rank']}")
        print(f"Chunk ID: {result['id']}")
        print(f"First line: {result['first_line'][:120]}")
        print(f"BM25 score: {result['score']:.4f}")
        print("-" * 45)

    retrieved_ids = {r["id"] for r in results}
    hit_at_3 = int(bool(retrieved_ids & relevant_ids))

    print(f"Relevant chunks: {sorted(relevant_ids)}")
    print(f"Hit@3: {hit_at_3}")


Q-01: What is form GR-305 used for?
Rank: 1
Chunk ID: GO-08
First line: Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used 
BM25 score: 13.4274
---------------------------------------------
Rank: 2
Chunk ID: GO-12
First line: A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using for
BM25 score: 8.7892
---------------------------------------------
Rank: 3
Chunk ID: GO-06
First line: Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is
BM25 score: 5.8155
---------------------------------------------
Relevant chunks: ['GO-08', 'GO-12', 'GO-23']
Hit@3: 1

Q-02: My family is poor. Is there financial help for students like me?
Rank: 1
Chunk ID: KZ-25
First line: Beshbarmak is usually made from finely chopped boiled meat, mixed with dough (typically egg noodles) and chyk, an onion 
BM25 score: 4

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [43]:
import os
import torch
from pathlib import Path
from dotenv import load_dotenv

cache_dir = Path(r"E:\hm3\hf_cache")
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(cache_dir)

from sentence_transformers import SentenceTransformer

load_dotenv(".env")
hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN is missing from .env")

model = SentenceTransformer(
    "google/embeddinggemma-300m",
    token=hf_token,
    cache_folder=str(cache_dir),
    device="cuda",
    model_kwargs={"torch_dtype": torch.float32}
)

doc_texts = [doc["text"] for doc in documents]

doc_embeddings = model.encode_document(
    doc_texts,
    batch_size=8,
    show_progress_bar=True,
    convert_to_tensor=True
)

print("Documents:", len(documents))
print("Embedding shape:", doc_embeddings.shape)

Batches: 100%|██████████| 10/10 [00:02<00:00,  4.79it/s]

Documents: 74
Embedding shape: torch.Size([74, 768])


In [44]:
def search_embedding(query, top_k=3):
    query_embedding = model.encode_query(
        query,
        convert_to_tensor=True
    )

    scores = model.similarity(
        query_embedding,
        doc_embeddings
    )[0]

    top_indices = sorted(
        range(len(documents)),
        key=lambda i: (-float(scores[i]), i)
    )[:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        doc = documents[idx]

        first_line = next(
            (line.strip() for line in doc["text"].splitlines()
             if line.strip()),
            ""
        )

        results.append({
            "rank": rank,
            "id": doc["id"],
            "first_line": first_line,
            "score": float(scores[idx])
        })

    return results


In [45]:
embedding_results = {}
embedding_hits = {}

for q in queries:
    query_id = q["id"]
    query_text = q["text"]
    relevant_ids = set(q["relevant"])

    results = search_embedding(query_text, top_k=3)
    embedding_results[query_id] = results

    print(f"\n{'=' * 65}")
    print(f"{query_id}: {query_text}")
    print(f"{'=' * 65}")

    for result in results:
        print(f"Rank: {result['rank']}")
        print(f"Chunk ID: {result['id']}")
        print(f"First line: {result['first_line'][:120]}")
        print(f"Cosine score: {result['score']:.4f}")
        print("-" * 45)

    retrieved_ids = {r["id"] for r in results}
    matched_ids = retrieved_ids & relevant_ids
    hit_at_3 = int(bool(matched_ids))

    embedding_hits[query_id] = hit_at_3

    print(f"Relevant chunks: {sorted(relevant_ids)}")
    print(f"Hit@3: {hit_at_3}")

print("\nEmbeddingGemma Hit@3:",
      f"{sum(embedding_hits.values())}/{len(embedding_hits)}")


Q-01: What is form GR-305 used for?
Rank: 1
Chunk ID: GO-08
First line: Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used 
Cosine score: 0.6955
---------------------------------------------
Rank: 2
Chunk ID: GO-12
First line: A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using for
Cosine score: 0.4609
---------------------------------------------
Rank: 3
Chunk ID: GO-23
First line: Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календарных дней со дня реше
Cosine score: 0.4505
---------------------------------------------
Relevant chunks: ['GO-08', 'GO-12', 'GO-23']
Hit@3: 1

Q-02: My family is poor. Is there financial help for students like me?
Rank: 1
Chunk ID: GO-01
First line: The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a stu
Cosine s

## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id   | Query                                                                                                    | `relevant`          | Expected winner, and why                                                                              |
|------|----------------------------------------------------------------------------------------------------------|---------------------|-------------------------------------------------------------------------------------------------------|
| Q-09 | GR-104                                                                                                   | GO-06               | BM25 perform better because the query contains only an exact form code, without any semantic context. |
| Q-10 | I am struggling to afford university because my family has very little money. What support is available? | GO-01, GO-02, GO-03 | EmbeddingGemma - semantic paraphrasing                                                                |


In [46]:
own_queries = [
    {
        "id": "Q-09",
        "text": "GR-104",
        "relevant": ["GO-06"]
    },
    {
        "id": "Q-10",
        "text": (
            "I am struggling to afford university because my "
            "family has very little money. "
            "What support is available?"
        ),
        "relevant": ["GO-01", "GO-02", "GO-03"]
    }
]

all_queries = queries + own_queries

print(f"Total queries: {len(all_queries)}")

for q in own_queries:
    print(f"\n{q['id']}: {q['text']}")
    print(f"Relevant: {q['relevant']}")


Total queries: 10

Q-09: GR-104
Relevant: ['GO-06']

Q-10: I am struggling to afford university because my family has very little money. What support is available?
Relevant: ['GO-01', 'GO-02', 'GO-03']


In [47]:
bm25_results = {}
embedding_results = {}
comparison = []

for q in all_queries:
    query_id = q["id"]
    query_text = q["text"]
    relevant = set(q["relevant"])

    bm_results = search_bm25(query_text, top_k=3)
    em_results = search_embedding(query_text, top_k=3)

    bm25_results[query_id] = bm_results
    embedding_results[query_id] = em_results

    bm_ids = {r["id"] for r in bm_results}
    em_ids = {r["id"] for r in em_results}

    bm_hit = int(bool(bm_ids & relevant))
    em_hit = int(bool(em_ids & relevant))

    comparison.append({
        "id": query_id,
        "bm25_top3": [r["id"] for r in bm_results],
        "embedding_top3": [r["id"] for r in em_results],
        "bm25_hit": bm_hit,
        "embedding_hit": em_hit
    })

    print(f"\n{'=' * 65}")
    print(f"{query_id}: {query_text}")
    print(f"{'=' * 65}")

    print("\nBM25 TOP-3:")
    for r in bm_results:
        print(f"{r['rank']}. {r['id']} | Score: {r['score']:.4f}")
        print(f"   {r['first_line'][:120]}")

    print("\nEMBEDDING TOP-3:")
    for r in em_results:
        print(f"{r['rank']}. {r['id']} | Cosine: {r['score']:.4f}")
        print(f"   {r['first_line'][:120]}")

    print(f"\nRelevant: {sorted(relevant)}")
    print(f"BM25 Hit@3: {bm_hit}")
    print(f"Embedding Hit@3: {em_hit}")

bm25_total = sum(r["bm25_hit"] for r in comparison)
embedding_total = sum(r["embedding_hit"] for r in comparison)

print("\n" + "=" * 65)
print("FINAL RESULTS")
print("=" * 65)

print(f"BM25 Hit@3: {bm25_total}/10")
print(f"EmbeddingGemma Hit@3: {embedding_total}/10")



Q-01: What is form GR-305 used for?

BM25 TOP-3:
1. GO-08 | Score: 13.4274
   Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used 
2. GO-12 | Score: 8.7892
   A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using for
3. GO-06 | Score: 5.8155
   Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is

EMBEDDING TOP-3:
1. GO-08 | Cosine: 0.6955
   Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used 
2. GO-12 | Cosine: 0.4609
   A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using for
3. GO-23 | Cosine: 0.4505
   Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календарных дней со дня реше

Relevant: ['GO-08', 'GO-12', 'GO-

## Results for all ten or more queries: top 3 from each retriever

In [48]:
import pandas as pd

comparison_df = pd.DataFrame(comparison)

comparison_df = comparison_df.rename(columns={
    "id": "Query",
    "bm25_top3": "BM25 Top-3",
    "embedding_top3": "Embedding Top-3",
    "bm25_hit": "BM25 Hit@3",
    "embedding_hit": "Embedding Hit@3"
})

display(comparison_df)

print("\nOverall:")
print(f"BM25: {bm25_total}/10 = {bm25_total * 10}%")
print(f"EmbeddingGemma: {embedding_total}/10 = {embedding_total * 10}%")


,Query,BM25 Top-3,Embedding Top-3,BM25 Hit@3,Embedding Hit@3
0,Q-01,"[GO-08, GO-12, GO-06]","[GO-08, GO-12, GO-23]",1,1
1,Q-02,"[KZ-25, KZ-13, KZ-17]","[GO-01, GO-10, GO-20]",0,1
2,Q-03,"[GO-14, KZ-01, KZ-02]","[GO-14, GO-09, GO-11]",1,1
3,Q-04,"[GO-05, GO-19, GO-17]","[GO-02, GO-05, GO-17]",0,1
4,Q-05,"[KZ-05, KZ-08, KZ-11]","[KZ-05, KZ-06, KZ-03]",1,1
5,Q-06,"[KZ-37, KZ-38, KZ-03]","[KZ-37, KZ-38, KZ-10]",1,1
6,Q-07,"[GO-21, GO-02, GO-06]","[GO-15, GO-18, GO-01]",0,1
7,Q-08,"[GO-17, GO-04, GO-01]","[GO-17, GO-18, GO-15]",1,1
8,Q-09,"[GO-06, GO-09, GO-14]","[GO-06, GO-14, GO-09]",1,1
9,Q-10,"[GO-16, KZ-44, KZ-07]","[GO-01, GO-16, GO-20]",0,1



Overall:
BM25: 6/10 = 60%
EmbeddingGemma: 10/10 = 100%


## B3 — comparison

| Query           | BM25 top 3 (ids)    | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|-----------------|---------------------|:----------:|-----------------------|:---------------:|
| Q-01            | GO-08, GO-12, GO-06 |     1      | GO-08, GO-12, GO-23   |        1        |
| Q-02            | KZ-25, KZ-13, KZ-17 |     0      | GO-01, GO-10, GO-20   |        1        |
| Q-03            | GO-14, KZ-01, KZ-02 |     1      | GO-14, GO-09, GO-11   |        1        |
| Q-04            | GO-05, GO-19, GO-17 |     0      | GO-02, GO-05, GO-17   |        1        |
| Q-05            | KZ-05, KZ-08, KZ-11 |     1      | KZ-05, KZ-06, KZ-03   |        1        |
| Q-06            | KZ-37, KZ-38, KZ-03 |     1      | KZ-37, KZ-38, KZ-10   |        1        |
| Q-07            | GO-21, GO-02, GO-06 |     0      | GO-15, GO-18, GO-01   |        1        |
| Q-08            | GO-17, GO-04, GO-01 |     1      | GO-17, GO-18, GO-15   |        1        |
| Q-09            | GO-06, GO-09, GO-14 |     1      | GO-06, GO-14, GO-09   |        1        |
| Q-10            | GO-16, KZ-44, KZ-07 |     0      | GO-01, GO-16, GO-20   |        1        |
| **hit@3 total** |                     |  **6/10**  |                       |    **10/10**    |

**Both retrievers agree:** which query, and why was it easy?

>Q-06: "Where did the Soviet Union test its nuclear weapons?"
>
>Both retrievers ranked KZ-37 and KZ-38 in the first two positions. The query was easy because the documents contain matching terms such as "Soviet Union" and "nuclear tests", and their meaning closely matches the question.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

>There was no query where BM25 performed better based on Hit@3. I expected BM25 to win on Q-09 because it could match the exact identifier "GR-104". However, both retrievers ranked GO-06 first and achieved Hit@3 = 1. Therefore, my original prediction was not confirmed.

**Embeddings did better:** which query, and what could BM25 not match?

>Q-07: "Can the grant be sent to my mother's card?"
>
>EmbeddingGemma retrieved the correct document GO-15 in first place, while BM25 failed to find it in the top three. The relevant information was written in Russian, but the question was in English. BM25 could not match the English wording with the Russian text, while EmbeddingGemma successfully captured the meaning across languages.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [49]:
import json
import tiktoken
from pathlib import Path

with open("data/qa.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

if isinstance(qa_data, list):
    qa_questions = qa_data
else:
    qa_questions = qa_data["questions"]

print("Loaded questions:", len(qa_questions))

selected = qa_questions[0]
question = selected["question"]

retrieved = search_embedding(question, top_k=3)

docs_by_id = {d["id"]: d for d in documents}

context_blocks = []

for i, result in enumerate(retrieved, start=1):
    chunk_id = result["id"]
    chunk_text = docs_by_id[chunk_id]["text"]

    block = f"[{i}] Chunk ID: {chunk_id}\n{chunk_text}"
    context_blocks.append(block)

context_text = "\n\n".join(context_blocks)

system_message = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, "
    "say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

context_part = f"Context:\n{context_text}"
question_part = f"\n\nQuestion:\n{question}"

user_message = context_part + question_part

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

Loaded questions: 6


## C1 — the complete prompt, printed before it is sent

In [50]:
print("\n" + "=" * 70)
print("COMPLETE PROMPT")
print("=" * 70)

for message in messages:
    print(f"\nROLE: {message['role']}")
    print(message["content"])


COMPLETE PROMPT

ROLE: system
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

ROLE: user
Context:
[1] Chunk ID: GO-04
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Chunk ID: GO-03
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their studen

In [51]:
encoding = tiktoken.get_encoding("o200k_base")

system_tokens = len(encoding.encode(system_message))
question_tokens = len(encoding.encode(question_part))
context_tokens = len(encoding.encode(context_part))

total_tokens = (
        system_tokens + question_tokens + context_tokens
)

print("TOKEN COUNT")
print("-" * 45)
print("System instructions:", system_tokens)
print("Question:", question_tokens)
print("Retrieved context:", context_tokens)
print("Total input estimate:", total_tokens)

print("\nCONTEXT DETAILS")
print("-" * 45)
print("Number of chunks:", len(retrieved))

for i, result in enumerate(retrieved, start=1):
    chunk_text = docs_by_id[result["id"]]["text"]
    tokens = len(encoding.encode(chunk_text))

    print(f"Chunk [{i}] ({result['id']}): {tokens} tokens")

print(
    "Total context size:",
    len(encoding.encode(context_text)),
    "tokens"
)

print(
    "Total context characters:",
    len(context_text)
)

TOKEN COUNT
---------------------------------------------
System instructions: 44
Question: 21
Retrieved context: 311
Total input estimate: 376

CONTEXT DETAILS
---------------------------------------------
Number of chunks: 3
Chunk [1] (GO-04): 90 tokens
Chunk [2] (GO-03): 104 tokens
Chunk [3] (GO-15): 85 tokens
Total context size: 309 tokens
Total context characters: 1193


In [52]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env")

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is missing")

client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages
)

answer = response.choices[0].message.content

print("MODEL ANSWER:")
print(answer)

print("\nTOKEN COMPARISON:")
print("Tiktoken estimate:", total_tokens)
print("API prompt tokens:", response.usage.prompt_tokens)
print(
    "Difference:",
    response.usage.prompt_tokens - total_tokens
)

MODEL ANSWER:
A student in income band 2 receives **150,000 tenge per academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

TOKEN COMPARISON:
Tiktoken estimate: 376
API prompt tokens: 385
Difference: 9


| Part of the input                           | Tokens (tiktoken `o200k_base`) |
|---------------------------------------------|-------------------------------:|
| System instructions                         |                             44 |
| Question                                    |                             21 |
| Retrieved context                           |                            311 |
| **Total input**                             |                            376 |
| **`usage.prompt_tokens` from the response** |                            385 |

- Number of retrieved chunks: 3
- Tokens per chunk: ~93
- Total context size (tokens / characters): 309 / 1193

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>The tiktoken count is an estimate based on the visible text using the `o200k_base` encoding. The API token count can differ because it also accounts for message formatting, roles, and special tokens. The model may also use a different tokenization scheme.

**What exactly does retrieval add to the model's input?**

>Retrieval adds the text of the three most relevant document chunks to the prompt as context. Each chunk includes its ID and a reference number. The LLM receives the retrieved text together with the system instructions and the question. It does not receive the embeddings or search the original documents itself.

## C2 — three systems, six questions

In [53]:
import json
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(".env")
client = OpenAI()

MODEL_NAME = "gpt-5.6-luna"

assert len(qa_questions) == 6

docs_by_id = {d["id"]: d for d in documents}

system_message = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, "
    "say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)


def build_rag_prompt(question, search_function):
    retrieved = search_function(question, top_k=3)

    blocks = []
    for i, result in enumerate(retrieved, start=1):
        chunk_id = result["id"]
        text = docs_by_id[chunk_id]["text"]
        blocks.append(f"[{i}] Chunk ID: {chunk_id}\n{text}")

    context = "\n\n".join(blocks)

    messages = [
        {"role": "system", "content": system_message},
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{question}"
        }
    ]

    return messages, retrieved, blocks


cache_path = Path("c2_results.json")

if cache_path.exists():
    with open(cache_path, "r", encoding="utf-8") as f:
        c2_results = json.load(f)
else:
    c2_results = {}

systems = ["llm_only", "bm25_rag", "embedding_rag"]

for q in qa_questions:
    qid = q["id"]
    question = q["question"]
    reference = q["reference"]

    if qid not in c2_results:
        c2_results[qid] = {
            "question": question,
            "reference": reference,
            "systems": {}
        }

    print(f"\n{'=' * 70}")
    print(f"{qid}: {question}")
    print(f"REFERENCE: {reference}")
    print(f"{'=' * 70}")

    for system in systems:

        if system in c2_results[qid]["systems"]:
            result = c2_results[qid]["systems"][system]
            print(f"\n{system.upper()} (cached)")
            print(result["answer"])
            continue

        if system == "llm_only":
            messages = [
                {"role": "user", "content": question}
            ]
            retrieved = []
            blocks = []

        elif system == "bm25_rag":
            messages, retrieved, blocks = build_rag_prompt(
                question, search_bm25
            )

        else:
            messages, retrieved, blocks = build_rag_prompt(
                question, search_embedding
            )

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages
        )

        answer = response.choices[0].message.content

        c2_results[qid]["systems"][system] = {
            "answer": answer,
            "retrieved": retrieved,
            "context_blocks": blocks,
            "messages": messages,
            "prompt_tokens": response.usage.prompt_tokens
        }

        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(
                c2_results, f,
                ensure_ascii=False, indent=2
            )

        print(f"\n{system.upper()}")
        print("Retrieved:", [r["id"] for r in retrieved])
        print("Answer:", answer)

print("\nCompleted C2 runs.")
print("Results saved to:", cache_path)


C-01: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

LLM_ONLY (cached)
Which student support scheme and academic year do you mean? “Income band 2” varies by country and provider, and the payment amount and schedule depend on that—for example, SUSI in Ireland, Student Finance, or a university bursary.

BM25_RAG (cached)
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1]

EMBEDDING_RAG (cached)
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments: 75,000 tenge in October and 75,000 tenge in February**. [1]

C-02: My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.

LLM

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question                  | LLM only                                     | BM25 RAG                                     | Embedding RAG                                |
|---------------------------|----------------------------------------------|----------------------------------------------|----------------------------------------------|
| C-01                      | ✗ Asked for clarification                    | ✓ 150,000 KZT, October and February          | ✓ 150,000 KZT, October and February          |
| C-02                      | ✗ Did not identify the deadline or form      | ✓ 14 calendar days, GR-305                   | ✓ 14 calendar days, GR-305                   |
| C-03                      | ✗ Could not identify the GPA requirement     | ✓ GPA 2.67/4.0                               | ✓ GPA 2.67/4.0                               |
| C-04                      | ✓ October 2017, completion in 2031           | ✓ October 2017, completion in 2031           | ✓ October 2017, completion in 2031           |
| C-05                      | ✓ 1963, Almaty Institute of National Economy | ✓ 1963, Almaty Institute of National Economy | ✓ 1963, Almaty Institute of National Economy |
| C-06                      | ✗ Gave unrelated US grant information        | ✓ Amount unknown, separate master's scheme   | ✓ Amount unknown, separate master's scheme   |
| **agrees with reference** | **2/6**                                      | **6/6**                                      | **6/6**                                      |

In [54]:
import json

with open("c2_results.json", "r", encoding="utf-8") as f:
    c2_results = json.load(f)

selected_questions = ["C-03", "C-04", "C-06"]

for qid in selected_questions:
    data = c2_results[qid]

    print("\n" + "=" * 70)
    print(qid, data["question"])
    print("REFERENCE:", data["reference"])
    print("=" * 70)

    for system in ["bm25_rag", "embedding_rag"]:
        result = data["systems"][system]

        print(f"\n--- {system.upper()} ---")

        print("\nRETRIEVED CHUNKS:")
        for r in result["retrieved"]:
            print(
                f"{r['rank']}. {r['id']} "
                f"| Score: {r['score']:.4f}"
            )

        print("\nCONTEXT SENT TO MODEL:")
        for block in result["context_blocks"]:
            print(block)
            print()

        print("\nMODEL ANSWER:")
        print(result["answer"])



C-03 What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

--- BM25_RAG ---

RETRIEVED CHUNKS:
1. GO-16 | Score: 13.9259
2. GO-19 | Score: 11.1301
3. GO-02 | Score: 9.7761

CONTEXT SENT TO MODEL:
[1] Chunk ID: GO-16
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] Chunk ID: GO-19
The grant budget is fixed each year. If there are more qualifying applicants than funds, grants are awarded in this order: band 1 applicants before band 2 applicants; within a band, the higher GPA first; and when two applicants have the same band and the same GPA, the earlier GR-104 submission ti

## C3 — question 1 of 3: (id)

**Question:** What is the minimum GPA needed to qualify for the need-based grant?

**1. What BM25 retrieved** (top 3, ids and scores):

| Rank | Chunk ID | BM25 Score |
|------|----------|-----------:|
| 1    | GO-16    |    13.9259 |
| 2    | GO-19    |    11.1301 |
| 3    | GO-02    |     9.7761 |

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

| Rank | Chunk ID | Cosine Score |
|------|----------|-------------:|
| 1    | GO-02    |       0.6286 |
| 2    | GO-20    |       0.5691 |
| 3    | GO-05    |       0.5451 |

**3. What was sent to the model** (the context blocks):

>BM25 context:
> - [1] GO-16: Scholarship compatibility and the 400,000 tenge support cap.
> - [2] GO-19: Applicant selection priorities based on income bands and GPA.
> - [3] GO-02: The 2026 grant eligibility requirements, including the minimum GPA of 2.67.

>Embedding context:
> - [1] GO-02: The current GPA requirement of 2.67.
> - [2] GO-20: Eligibility restrictions for undergraduate students.
> - [3] GO-05: The archived 2025 rules with a minimum GPA of 3.0.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Without RAG, the LLM could not identify the GPA requirement and asked for more information.
>
>Both RAG systems answered correctly: 2.67 on a 4.0 scale.
>
>BM25 found the correct information only in its third chunk, while EmbeddingGemma ranked it first. EmbeddingGemma also retrieved an outdated rule, but the LLM correctly used the current information.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>Neither system failed to answer the question.
>
>GO-02 explicitly states that the "cumulative GPA is at least 2.67 on the 4.0 scale."
>
>EmbeddingGemma also retrieved GO-05, which contains an outdated GPA of 3.0. However, that chunk is clearly marked "ARCHIVED — kept for reference only."
>
>The model correctly distinguished the current and outdated rules. There was some irrelevant retrieval, but no generation failure.

In [55]:
import json

with open("c2_results.json", "r", encoding="utf-8") as f:
    c2_results = json.load(f)

def show_c3_context(qid):
    data = c2_results[qid]

    print(f"Question: {data['question']}")
    print(f"Reference: {data['reference']}")

    for system in ["bm25_rag", "embedding_rag"]:
        result = data["systems"][system]

        print("\n" + "=" * 65)
        print(system.upper())
        print("=" * 65)

        print("\nRETRIEVED CHUNKS:")
        for r in result["retrieved"]:
            print(
                f"{r['rank']}. {r['id']} "
                f"| Score: {r['score']:.4f}"
            )

        print("\nFULL CONTEXT BLOCKS:")
        print("\n\n".join(result["context_blocks"]))

        print("\nMODEL ANSWER:")
        print(result["answer"])

show_c3_context("C-03")

Question: What is the minimum GPA needed to qualify for the need-based grant?
Reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

BM25_RAG

RETRIEVED CHUNKS:
1. GO-16 | Score: 13.9259
2. GO-19 | Score: 11.1301
3. GO-02 | Score: 9.7761

FULL CONTEXT BLOCKS:
[1] Chunk ID: GO-16
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] Chunk ID: GO-19
The grant budget is fixed each year. If there are more qualifying applicants than funds, grants are awarded in this order: band 1 applicants before band 2 applicants; within a band, the higher GPA first; and when two applicants have the same band and the same GPA, the earlier GR-104 submission time win

## C3 — question 2 of 3: (id)

**Question:** When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?

**1. What BM25 retrieved** (top 3, ids and scores):

| Rank | Chunk ID | BM25 Score |
|------|----------|-----------:|
| 1    | KZ-43    |    18.7693 |
| 2    | KZ-14    |    12.5391 |
| 3    | GO-22    |     4.3842 |

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

| Rank | Chunk ID | Cosine Score |
|------|----------|-------------:|
| 1    | KZ-14    |       0.7204 |
| 2    | KZ-43    |       0.6927 |
| 3    | KZ-05    |       0.4342 |

**3. What was sent to the model** (the context blocks):

>BM25 context:
> - [1] KZ-43: The October 2017 decree and the extension of the transition deadline to 2031.
> - [2] KZ-14: Details about the alphabet reform and the original 2025 target.
> - [3] GO-22: Grant Office data retention rules, unrelated to the question.

> Embedding context:
> - [1] KZ-14: Details about the alphabet reform and its implementation.
> - [2] KZ-43: The 2017 decree and the revised 2031 deadline.
> - [3] KZ-05: Information about Astana, unrelated to the question.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>The LLM-only system already answered correctly using its existing knowledge.
>
>BM25 RAG also answered correctly and included the original 2025 deadline and its extension to 2031 in January 2021.
>
>Embedding RAG correctly answered October 2017 and 2031, but its answer was less detailed. It did not mention the original 2025 deadline or the January 2021 revision.
>
>This was slightly worse than I expected because both retrieved relevant chunks contained these details.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>Both retrievers found the necessary information.
>
>KZ-43 states that the transition was originally planned for 2025 and that "the target year for finishing the transition was pushed back to 2031."
>
>However, Embedding RAG omitted the original deadline and the date of the revision.
>
>This is a minor generation-level omission, not a retrieval failure, because the information was present in the context. The main answer was still correct.
>
>Both retrievers also included one irrelevant chunk, showing that retrieval can introduce unnecessary context.

In [56]:
show_c3_context("C-04")

Question: When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
Reference: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.

BM25_RAG

RETRIEVED CHUNKS:
1. KZ-43 | Score: 18.7693
2. KZ-14 | Score: 12.5391
3. GO-22 | Score: 4.3842

FULL CONTEXT BLOCKS:
[1] Chunk ID: KZ-43
The Kazakh language has been written mainly in four scripts at various points of time—Old Turkic, Cyrillic, Latin, and Arabic—each having a distinct alphabet. The Arabic script is used in Iran, Afghanistan, and China, while the Cyrillic script is used in Kazakhstan, Kyrgyzstan, Russia, and Mongolia. In October 2017, a presidential decree in Kazakhstan ordered a transition from the Cyrillic to Latin script to be implemented by 2025. In January 2021, the target year for finishing the transition was pushed back to 2031.

[2] Chunk ID: KZ-14
In October 2017, the then president, Nursultan Naza

## C3 — question 3 of 3: (id)

**Question:** I am a master's student. How much need-based grant can I get?

**1. What BM25 retrieved (top 3, ids and scores):**

| Rank | Chunk ID | BM25 Score |
|------|----------|-----------:|
| 1    | GO-20    |    13.6874 |
| 2    | GO-16    |    11.4071 |
| 3    | GO-01    |     9.0369 |

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

| Rank | Chunk ID | Cosine Score |
|------|----------|-------------:|
| 1    | GO-20    |       0.6664 |
| 2    | GO-16    |       0.6161 |
| 3    | GO-01    |       0.5539 |

**3. What was sent to the model (the context blocks):**

> Both retrievers returned the same three chunks in the same order:
>
> - [1] GO-20: The grant is only for undergraduate students. Master's students have a separate scheme.
> - [2] GO-16: Scholarship compatibility and the 400,000 tenge university support cap.
> - [3] GO-01: General information about the need-based undergraduate grant.

The complete context blocks are printed in the code cell below.

**4. How retrieval changed the answer:**

> Without RAG, the LLM gave a general answer about financial aid in the United States. It mentioned Pell Grants and other programs unrelated to this Grant Office.
>
>Both RAG systems correctly stated that the amount for master's students is unknown because the handbook does not describe their separate grant scheme.
>
>Retrieval made the answers more grounded and prevented the model from inventing an amount.

**5. What was missed, and was it a retrieval problem or a generation problem?**

>The corpus does not contain the master's grant amount.
>
>GO-20 states that "Master's and doctoral students are supported by a separate scheme, which this handbook does not describe."
>
>Therefore, neither retriever could provide the amount. This was not a retrieval failure because the required information was absent from the corpus.
>
>Both RAG systems handled this correctly by saying that they did not know the amount.
>
>There was also no generation failure because neither model invented a number from the unrelated grant information.

In [57]:
show_c3_context("C-06")

Question: I am a master's student. How much need-based grant can I get?
Reference: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.

BM25_RAG

RETRIEVED CHUNKS:
1. GO-20 | Score: 13.6874
2. GO-16 | Score: 11.4071
3. GO-01 | Score: 9.0369

FULL CONTEXT BLOCKS:
[1] Chunk ID: GO-20
The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules.

[2] Chunk ID: GO-16
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that th

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

>BM25 uses keyword matching and term frequencies to rank documents, while embedding retrieval compares numerical vector representations of their meaning. BM25 depends more on matching words, whereas embeddings can find relevant documents even when different words or languages are used.

**2. When can BM25 perform better than embeddings?**

>BM25 can perform better when a query contains exact identifiers, technical terms, or rare keywords. It is also generally faster and requires fewer computational resources. However, in my experiments, BM25 did not outperform EmbeddingGemma on Hit@3.

**3. When can embeddings perform better than BM25?**

>Embeddings perform better when queries use synonyms, paraphrases, or different languages. For example, in Q-07, EmbeddingGemma found the Russian document GO-15 using an English question, while BM25 failed to retrieve it.

**4. Why is cosine similarity used with embeddings?**

>Cosine similarity measures how closely two vectors point in the same direction rather than comparing their absolute magnitudes. This allows us to compare the semantic similarity between query and document embeddings even when their vector lengths differ.

**5. What information is actually sent to an LLM in a RAG system?**

>The LLM receives the system instructions, the user's question, and the text of the retrieved document chunks. In my implementation, each chunk also includes its ID and reference number. The model receives the retrieved text, not the embedding vectors or the complete document collection.

**6. Does the LLM search the original documents itself?**

>No. The retrieval system searches the documents and selects the relevant chunks before the LLM generates an answer. The LLM only sees the context included in its prompt.

**7. What happens if retrieval returns an irrelevant chunk?**

>An irrelevant chunk adds unnecessary information to the prompt and can distract the model or lead to an incorrect answer. For example, BM25 retrieved GO-22 in C-04, even though it describes Grant Office data retention rather than the Kazakh alphabet. The model still answered correctly, but this shows how irrelevant context can enter the generation process.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>I tested BM25 RAG on C-03 with K = 1, 3, and 8.
>
>| Top-K | Model Answer | Prompt Tokens |
>|---|---|---:|
>| 1 | Minimum GPA not found | 153 |
>| 3 | Correct: 2.67 | 357 |
>| 8 | Correct: 2.67 | 827 |
>
> With K = 1, the model could not answer because the retrieved document GO-16 did not contain the GPA requirement. With K = 3, the correct document GO-02 was included, and the model answered 2.67.
>
>Increasing K to 8 did not improve the answer but increased the input from 357 to 827 tokens. It also introduced the archived document GO-05 with an outdated GPA of 3.0, which could potentially confuse the model. However, in this experiment, the model still answered correctly.
>
>This shows that increasing Top-K can improve retrieval coverage, but adding too many chunks increases token usage and the risk of irrelevant or conflicting information.

In [58]:
import json
from pathlib import Path

question_id = "C-03"

qa = next(
    q for q in qa_questions
    if q["id"] == question_id
)

question = qa["question"]
reference = qa["reference"]

docs_by_id = {d["id"]: d for d in documents}

cache_path = Path("topk_results.json")

if cache_path.exists():
    with open(cache_path, "r", encoding="utf-8") as f:
        topk_results = json.load(f)
else:
    topk_results = {}

for k in [1, 3, 8]:
    cache_key = f"{question_id}_BM25_K{k}"

    if cache_key not in topk_results:
        retrieved = search_bm25(question, top_k=k)

        blocks = []
        for i, result in enumerate(retrieved, 1):
            chunk_id = result["id"]
            chunk_text = docs_by_id[chunk_id]["text"]

            blocks.append(
                f"[{i}] Chunk ID: {chunk_id}\n{chunk_text}"
            )

        context = "\n\n".join(blocks)

        messages = [
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context}\n\n"
                    f"Question:\n{question}"
                )
            }
        ]

        response = client.chat.completions.create(
            model="gpt-5.6-luna",
            messages=messages
        )

        topk_results[cache_key] = {
            "k": k,
            "retrieved": retrieved,
            "answer": response.choices[0].message.content,
            "prompt_tokens": response.usage.prompt_tokens
        }

        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(
                topk_results,
                f,
                indent=2,
                ensure_ascii=False
            )

    result = topk_results[cache_key]

    print("\n" + "=" * 65)
    print(f"TOP-K EXPERIMENT: K = {k}")
    print("=" * 65)

    print("Question:", question)
    print("Reference:", reference)

    print("\nRetrieved chunks:")
    for doc in result["retrieved"]:
        print(f"{doc['id']} | Score: {doc['score']:.4f}")

    print("\nModel answer:")
    print(result["answer"])

    print("\nPrompt tokens:", result["prompt_tokens"])



TOP-K EXPERIMENT: K = 1
Question: What is the minimum GPA needed to qualify for the need-based grant?
Reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

Retrieved chunks:
GO-16 | Score: 13.9259

Model answer:
The context does not specify a minimum GPA for the need-based grant.

Prompt tokens: 153

TOP-K EXPERIMENT: K = 3
Question: What is the minimum GPA needed to qualify for the need-based grant?
Reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

Retrieved chunks:
GO-16 | Score: 13.9259
GO-19 | Score: 11.1301
GO-02 | Score: 9.7761

Model answer:
The minimum GPA needed is **2.67 on a 4.0 scale**. [3]

Prompt tokens: 357

TOP-K EXPERIMENT: K = 8
Question: What is the minimum GPA needed to qualify for the need-based grant?
Reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

Retrieved chunks:
GO-16 | Score: 13.9259

**9. Is a vector database required to build a RAG system?**

>No. In my implementation, the embeddings were stored directly in memory as a PyTorch tensor with shape [74, 768]. Cosine similarity was computed against these vectors without using a vector database. A vector database becomes more useful for larger collections and efficient similarity search.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>EmbeddingGemma performed better in retrieval, achieving Hit@3 of 10/10 (100%), compared with BM25's 6/10 (60%). However, both RAG systems achieved 6/6 correct answers in Part C2. This shows that better retrieval results do not always produce higher answer accuracy on a small test set.

---
# Conclusion

>In this homework, I compared keyword-based and semantic retrieval and tested how they affect LLM answers. EmbeddingGemma retrieved relevant documents more consistently, especially for paraphrased and multilingual queries, while BM25 was simpler and less computationally expensive. Both RAG systems improved answer accuracy compared with using the LLM alone. Next, I would build a hybrid retrieval system combining BM25 and embeddings, add filtering for outdated documents, and evaluate its accuracy and response time on a larger set of questions.